# Text-to-SQL Fine-Tuning Pipeline (Qwen2.5-Coder-7B + BIRD/Spider)

Runs M1→M4 on Colab: **data prep → baseline → QLoRA fine-tuning → evaluation**.

- Base model: `Qwen2.5-Coder-7B-Instruct` (locked); method: BIRD(evidence) + Spider
- Source is packaged separately: build `text2sql.zip` locally, then upload in this notebook
- Design doc: `docs/TEXT2SQL_FINETUNE_DESIGN.md` (included in the zip)

> Note: choose a **T4 GPU** runtime (free) or L4/A100 (Pro+). All files are
> session-scoped and disappear on reset; uncomment the Drive-mount cell to persist.

## 0. Before you start (locally, once)

```bash
# Generate text2sql.zip from the repo root (contains data/, src/, requirements.txt, docs/)
python tools/build_text2sql_zip.py
```

Then prepare this notebook and `text2sql.zip` together (the zip is uploaded in step 3).

## 1. Run top-to-bottom

| Step | Command | Est. time |
|---|---|---|
| M1 data | `python data/prep.py` | ~5-10 min (downloads HF datasets) |
| M2 Spider DBs | `python data/download_spider_dbs.py` | ~1-3 min |
| M2 baseline | `python -m src.baseline_eval` | ~10 min for 50 samples; 1-2h for 500 (T4) |
| M3 training | `python -m src.train` | 1-2h full (A100) / several hours (T4) |
| M4 eval | `python -m src.eval` | same as baseline |

The **configuration** cell controls eval sample count, training smoke params, and the optional LLM semantic judge.

In [ ]:
# 2. Install dependencies (torch is preinstalled; training does not depend on TRL)
!pip install -q "datasets>=2.19" "sqlglot>=23" "transformers>=4.44" \
               accelerate bitsandbytes gdown openai "peft>=0.12" unsloth
!python -c "import sqlglot, datasets; print('deps OK:', sqlglot.__version__, datasets.__version__)" 

## 3. Upload and extract the source package

Run `python tools/build_text2sql_zip.py` locally to produce `text2sql.zip`,
then run the cell below to upload it. After extraction the working directory is
switched to `/content/text2sql`.

In [ ]:
from google.colab import files
print('Upload text2sql.zip')
uploaded = files.upload()
import os, zipfile

os.makedirs('/content/text2sql', exist_ok=True)
for fn in uploaded.keys():
    with zipfile.ZipFile(fn) as z:
        z.extractall('/content/text2sql')

os.chdir('/content/text2sql')
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))
# quick sanity check: required files present
missing = [f for f in ('data/prep.py', 'src/train.py', 'src/baseline_eval.py', 'requirements.txt')
           if not os.path.exists(f)]
assert not missing, f"zip missing required files: {missing}; repackage and re-upload" 

In [ ]:
# 4. GPU check (must be a GPU runtime)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))

## 5. Configuration

- `LIMIT_BASELINE`: baseline/eval sample count. **Use 50 for a smoke run, then 500 for real numbers**.
- `LIMIT_TRAIN` / `MAX_STEPS`: training smoke params (e.g. `LIMIT_TRAIN=200, MAX_STEPS=5`).
- `RUN_JUDGE`: whether to add the independent LLM semantic judge (needs a DeepSeek/Qwen API key).

In [ ]:
# 5. Configuration
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce OOM fragmentation
os.environ.setdefault("PYTHONUNBUFFERED", "1")  # stream progress lines live (no block buffering)

LIMIT_BASELINE = 50          # smoke: 50; real eval: 500 (~1-2h on T4)
LIMIT_TRAIN    = None        # training sample cap (None = full); smoke: 200
MAX_STEPS      = None        # early-stop steps (None = normal); smoke: 5
BATCH_SIZE     = 1           # T4-friendly; use 2-4 on A100
MAX_SEQ_LENGTH = 1536        # T4-friendly; use 2048 on A100 (drops fewer long schemas)
SKIP_TRAIN     = False       # True skips M3 training (data+baseline only)
RUN_JUDGE      = False       # whether to run the independent LLM semantic judge
JUDGE_API_KEY  = None        # DeepSeek key (or set OPENAI_API_KEY env var)
JUDGE_API_BASE = "https://api.deepseek.com"
JUDGE_MODEL    = "deepseek-chat"

if RUN_JUDGE and not JUDGE_API_KEY:
    raise ValueError("RUN_JUDGE=True but no JUDGE_API_KEY provided")

## 6. M1 — Data prep (download/clean/dedupe/split/contamination check)

Produces `data/processed/{train,val,test}.jsonl` + **`bird_dev.jsonl`** (the hard eval set) + `meta.json`. Progress is streamed live.

In [ ]:
!python -u data/prep.py

## 7. M2 — Download Spider databases (needed for execution eval)

Produces `data/spider_database/`.

In [ ]:
!python -u data/download_spider_dbs.py

## 8. M2 — Zero-shot baseline

Scores the **un-fine-tuned** model on the held-out set (Spider validation subset,
`LIMIT_BASELINE` samples). Results -> `results/baseline.json` (per-sample gold/pred
for M4 comparison).

In [ ]:
import subprocess

# -u = unbuffered: loss/step lines stream live into the cell
cmd = ["python", "-u", "-m", "src.baseline_eval",
       "--db-root", "data/spider_database",
       "--limit", str(LIMIT_BASELINE)]
if RUN_JUDGE:
    cmd += ["--judge", "--judge-api-key", JUDGE_API_KEY,
            "--judge-api-base", JUDGE_API_BASE, "--judge-model", JUDGE_MODEL]
print("$", " ".join(cmd), flush=True)
rc = subprocess.run(cmd).returncode
if rc != 0:
    print(f"command failed (exit {rc}) — scroll up for the full error")
    raise SystemExit(rc)

## 8b. M2b — (optional) baseline on BIRD dev (the hard set)

BIRD dev is where fine-tuning should show **real** gains (evidence + complex
multi-table queries). Requires M1 to have generated `bird_dev.jsonl`.
Execution accuracy is skipped (needs the official BIRD dev DBs); EM / validity /
semantic judge still work. Results -> `results/baseline_birddev.json`.

In [ ]:
import subprocess

cmd = ["python", "-u", "-m", "src.baseline_eval",
       "--test-jsonl", "data/processed/bird_dev.jsonl",
       "--out", "results/baseline_birddev.json",
       "--limit", str(LIMIT_BASELINE)]
if RUN_JUDGE:
    cmd += ["--judge", "--judge-api-key", JUDGE_API_KEY,
            "--judge-api-base", JUDGE_API_BASE, "--judge-model", JUDGE_MODEL]
print("$", " ".join(cmd), flush=True)
rc = subprocess.run(cmd).returncode
if rc != 0:
    print(f"command failed (exit {rc}) — scroll up for the full error")
    raise SystemExit(rc)

## 9. M3 — QLoRA fine-tuning (Unsloth 4-bit + completion-only loss)

Produces `outputs/lora/` (adapter + train_config.json + history.json).

Live progress: **every 10 steps** the trainer prints `loss`/`grad_norm`/`learning_rate`; eval every 100 steps prints `eval_loss`; a tqdm bar shows step/ETA.

In [ ]:
import subprocess

if not SKIP_TRAIN:
    cmd = ["python", "-u", "-m", "src.train",
           "--batch-size", str(BATCH_SIZE),
           "--max-seq-length", str(MAX_SEQ_LENGTH)]
    if LIMIT_TRAIN: cmd += ["--limit", str(LIMIT_TRAIN)]
    if MAX_STEPS:   cmd += ["--max-steps", str(MAX_STEPS)]
    print("$", " ".join(cmd), flush=True)
    rc = subprocess.run(cmd).returncode
    if rc != 0:
        print(f"command failed (exit {rc}) — scroll up for the full error")
        raise SystemExit(rc)
else:
    print("SKIP_TRAIN=True; skipping training (M4 eval will not run)")

## 10. M4 — Fine-tuned evaluation (same harness, same held-out as baseline)

Produces `results/finetuned.json` + `results/comparison.json` (baseline/finetuned/delta).

In [ ]:
import subprocess

if not SKIP_TRAIN:
    cmd = ["python", "-u", "-m", "src.eval",
           "--adapter", "outputs/lora",
           "--db-root", "data/spider_database",
           "--limit", str(LIMIT_BASELINE)]
    if RUN_JUDGE:
        cmd += ["--judge", "--judge-api-key", JUDGE_API_KEY,
                "--judge-api-base", JUDGE_API_BASE, "--judge-model", JUDGE_MODEL]
    print("$", " ".join(cmd), flush=True)
    rc = subprocess.run(cmd).returncode
    if rc != 0:
        print(f"command failed (exit {rc}) — scroll up for the full error")
        raise SystemExit(rc)
else:
    print("SKIP_TRAIN=True; skipping eval")

## 10b. M4b — (optional) fine-tuned eval on BIRD dev (the hard set)

Compares `results/baseline_birddev.json` vs `results/finetuned_birddev.json`
via `results/comparison_birddev.json`. **This is where BIRD training should
show real gains** (evidence + complex multi-table queries), unlike the easy
Spider-validation set.

In [ ]:
import subprocess

if not SKIP_TRAIN:
    cmd = ["python", "-u", "-m", "src.eval",
           "--adapter", "outputs/lora",
           "--test-jsonl", "data/processed/bird_dev.jsonl",
           "--baseline", "results/baseline_birddev.json",
           "--out", "results/finetuned_birddev.json",
           "--comparison", "results/comparison_birddev.json",
           "--limit", str(LIMIT_BASELINE)]
    if RUN_JUDGE:
        cmd += ["--judge", "--judge-api-key", JUDGE_API_KEY,
                "--judge-api-base", JUDGE_API_BASE, "--judge-model", JUDGE_MODEL]
    print("$", " ".join(cmd), flush=True)
    rc = subprocess.run(cmd).returncode
    if rc != 0:
        print(f"command failed (exit {rc}) — scroll up for the full error")
        raise SystemExit(rc)
else:
    print("SKIP_TRAIN=True; skipping BIRD-dev eval")

## 11. Interpreting results

`results/comparison.json` structure:

```json
{
  "baseline":  {"exact_match": 0.04, "sql_validity": 0.99, "execution_accuracy": 0.12},
  "finetuned": {"exact_match": 0.60, "sql_validity": 0.99, "execution_accuracy": 0.48},
  "delta":     {"exact_match": 0.56, "sql_validity": 0.00, "execution_accuracy": 0.36}
}
```

- **exact_match**: lower bound (penalizes different-but-correct SQL);
- **execution_accuracy**: primary metric (identical result sets);
- delta > 0 means fine-tuning helped; focus on **execution_accuracy gains** (not EM).
- **semantic_equiv** appears only when RUN_JUDGE=True.

Next (M5): merge to 16-bit + GGUF quantization + push to HF Hub (`src/publish.py`, not yet implemented).

In [ ]:
# 12. View results
import json, os

comp = "results/comparison.json"
if os.path.exists(comp):
    c = json.load(open(comp))
    print("=== comparison ===")
    for k, v in c["delta"].items():
        print(f"  {k:>18s}: {c['baseline'].get(k)} -> {c['finetuned'].get(k)}  (delta {v:+.4f})")
else:
    print("no comparison.json yet (M4 may not have run)")

base = "results/baseline.json"
if os.path.exists(base):
    b = json.load(open(base))
    print("\n=== baseline sample examples ===")
    for s in b["samples"][:2]:
        print("- question:", s["question"][:80])
        print("  gold:", s["gold"][:100])
        print("  pred:", s["pred"][:100])